<a href="https://colab.research.google.com/github/Harshada900/pyspark-practise/blob/main/18_Delta_Lake_MERGE_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install pyspark delta-spark

In [1]:
#!pip install delta-spark
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
import pyspark.sql.functions as f

In [2]:
builder = SparkSession.builder.appName("Delta") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:

# Existing data in Delta table (target)
existing_data = [
    (1, "Raj",   "Engineering", 50000),
    (2, "Priya", "Marketing",   70000),
    (3, "Amit",  "Engineering", 60000)
]

In [4]:

existing_df = spark.createDataFrame(existing_data, ["id", "name", "dept", "salary"])

In [5]:
existing_df.write.format("delta").mode("overwrite").save("/content/delta/employees")

In [6]:
# New incoming data (source)
new_data = [
    (2, "Priya", "Marketing",  75000),  # exists → salary updated
    (3, "Amit",  "HR",         65000),  # exists → dept + salary updated
    (4, "Sara",  "HR",         80000),  # new → insert
    (5, "Karan", "Marketing",  45000)   # new → insert
]
new_df = spark.createDataFrame(new_data, ["id", "name", "dept", "salary"])

## **Performing MERGE**

In [7]:
from delta.tables import DeltaTable

#load existing delta table

delta_table = DeltaTable.forPath(spark, "/content/delta/employees")


In [8]:
#MERGE

delta_table.alias("target").merge(
    new_df.alias("source"),
    "target.id=source.id" #match condition
).whenMatchedUpdate(set = { #if match found then update
    "name": "source.name",
    "dept": "source.dept",
    "salary": "source.salary"
}).whenNotMatchedInsert(values ={ #if match not found then insert
    "id": "source.id",
    "name": "source.name",
    "dept": "source.dept",
    "salary": "source.salary"
}).execute()


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [9]:
#check result

spark.read.format("delta").load("/content/delta/employees").orderBy("id").show()

+---+-----+-----------+------+
| id| name|       dept|salary|
+---+-----+-----------+------+
|  1|  Raj|Engineering| 50000|
|  2|Priya|  Marketing| 75000|
|  3| Amit|         HR| 65000|
|  4| Sara|         HR| 80000|
|  5|Karan|  Marketing| 45000|
+---+-----+-----------+------+



In SQL

In [10]:
'''
spark.sql("""
    MERGE INTO delta.`/content/delta/employees` AS target
    USING new_updates AS source
    ON target.id = source.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
'''

'\nspark.sql("""\n    MERGE INTO delta.`/content/delta/employees` AS target\n    USING new_updates AS source\n    ON target.id = source.id\n    WHEN MATCHED THEN UPDATE SET *\n    WHEN NOT MATCHED THEN INSERT *\n""")\n'

## **whenMatchedUpdateAll() and whenMatcheInsertAll()**

In [11]:
DeltaTable.forPath(spark, "/content/delta/employees") \
    .alias("t").merge(new_df.alias("s"), "t.id = s.id") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()    # upserts all cols !

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

History

In [12]:
DeltaTable.forPath(spark, "/content/delta/employees").history().show()

+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      6|2026-06-29 09:50:...|  NULL|    NULL|    MERGE|{predicate -> ["(...|NULL|    NULL|     NULL|          5|  Serializable|        false|{numTargetRowsCop...|        NULL|Apache-Spark/4.0....|
|      5|2026-06-29 09:50:...|  NULL|    NULL|    MERGE|{predicate -> ["(...|NULL|    NULL|     NULL|          4|  Serializable|        false|{numTargetRowsCop...|        NULL|Apache-Spark/4.0....|
|      4|2

#time travel

In [13]:
spark.read.format("delta").option("versionAsOf", 0).load("/content/delta/employees").show()

+---+-----+-----------+------+
| id| name|       dept|salary|
+---+-----+-----------+------+
|  2|Priya|  Marketing| 70000|
|  3| Amit|Engineering| 60000|
|  1|  Raj|Engineering| 50000|
+---+-----+-----------+------+



Restore

In [14]:
#before
spark.read.format("delta").load("/content/delta/employees").show()

+---+-----+-----------+------+
| id| name|       dept|salary|
+---+-----+-----------+------+
|  2|Priya|  Marketing| 75000|
|  3| Amit|         HR| 65000|
|  4| Sara|         HR| 80000|
|  5|Karan|  Marketing| 45000|
|  1|  Raj|Engineering| 50000|
+---+-----+-----------+------+



In [15]:
DeltaTable.forPath(spark, "/content/delta/employees").restoreToVersion(0)

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]

In [16]:
#after
spark.read.format("delta").load("/content/delta/employees").show()

+---+-----+-----------+------+
| id| name|       dept|salary|
+---+-----+-----------+------+
|  2|Priya|  Marketing| 70000|
|  3| Amit|Engineering| 60000|
|  1|  Raj|Engineering| 50000|
+---+-----+-----------+------+

